In [1]:
import os
os.chdir('../..')
folder = 'trained_models/mean_embeddings'
os.getcwd()

'/Users/nkapila6/Code/nlpvise/src'

In [2]:
from skorch import NeuralNetClassifier

from training_pipeline import TrainingPipeline
from models.convlstm import ConvLSTM
from models.convnet import ConvNet
from models.bilstm import BiLSTM
from models.mlp import mlp

In [3]:
training_pipeline = TrainingPipeline()
training_pipeline.CLS = False
training_pipeline.load_data()
evaluation_results = {}


Loading mean embeddings. CLS is False.
Data loaded successfully.
Train/Test split completed.


In [4]:
def evaluate_model(module, param_path):
    net = NeuralNetClassifier(module=module)
    net.initialize()
    net.load_params(f_params=param_path)
    net.module_.to(training_pipeline.device)

    return training_pipeline.eval_model(net, threshold=0.5)

In [5]:
convnet = None
convlstm = None
bilstm = None

with os.scandir(folder) as entries:
    for entry in entries:
        print(entry.name)
        match entry.name:
            case 'BiLSTM_MEAN.pkl':
                results = evaluate_model(module=BiLSTM(input_size=768), param_path=entry.path)
                evaluation_results['BiLSTM'] = results
            case 'mlp_MEAN.pkl':
                results = evaluate_model(module=mlp(input_size=768), param_path=entry.path)
                evaluation_results['MLP'] = results
            case 'ConvLSTM_MEAN.pkl':
                results = evaluate_model(module=ConvLSTM(input_size=768), param_path=entry.path)
                evaluation_results['ConvLSTM'] = results
            case 'ConvNet_MEAN.pkl':
                results = evaluate_model(module=ConvNet(input_size=768), param_path=entry.path)
                evaluation_results['ConvNet'] = results

.DS_Store
mlp_MEAN.pkl
ConvNet_MEAN.pkl
BiLSTM_MEAN.pkl
ConvLSTM_MEAN.pkl


In [6]:
import pandas as pd
pd.DataFrame(evaluation_results).T

,ACC,MCC,F1,AUPRC,AUROC
MLP,0.779933,0.232115,0.711041,0.614960,0.736629
ConvNet,0.784241,0.257642,0.717762,0.636079,0.753941
BiLSTM,0.791351,0.290878,0.722682,0.650777,0.767450
ConvLSTM,0.786639,0.259279,0.720549,0.636450,0.757668
